# ⚡ Aula 3 – GPUs em Python e Aplicações em Engenharia

## Computação de Alto Desempenho em Python para Engenharia Civil

**Objetivos desta aula:**
- Introduzir paralelismo massivo com GPU
- Usar CuPy (NumPy para GPU) e Numba CUDA
- Aplicar HPC a simulações reais (difusão de calor)
- Comparar performance CPU vs GPU

---

### 🎯 CPU vs GPU: Filosofias Diferentes

**CPU (Central Processing Unit):**
- Poucos núcleos complexos (4-64 cores)
- Otimizada para latência (velocidade individual)
- Hierarquia de cache complexa
- Ideal para código sequencial e ramificado

**GPU (Graphics Processing Unit):**
- Milhares de núcleos simples (1000+ cores)
- Otimizada para throughput (trabalho total)
- Arquitetura SIMD (Single Instruction, Multiple Data)
- Ideal para paralelismo massivo e dados regulares

### 🏗️ Aplicações GPU em Engenharia Civil
- **Simulações CFD** (Computational Fluid Dynamics)
- **Análise modal** de estruturas complexas
- **Processamento de imagens** (inspeção, monitoramento)
- **Machine Learning** para predição de falhas

In [1]:
# Import Required Libraries
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import multiprocessing as mp
import sys
import subprocess

# Tentar importar bibliotecas GPU com diagnóstico detalhado
def check_cuda_installation():
    """Verifica instalação CUDA e diagnóstica problemas"""
    print("🔍 Diagnóstico CUDA:")
    
    # Verificar CUDA Toolkit
    try:
        result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
        if result.returncode == 0:
            print(f"✅ NVCC encontrado: {result.stdout.split('release')[1].split(',')[0].strip()}")
        else:
            print("❌ NVCC não encontrado no PATH")
    except FileNotFoundError:
        print("❌ CUDA Toolkit não instalado ou não está no PATH")
    
    # Verificar driver NVIDIA
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ Driver NVIDIA funcionando")
            # Extrair versão do driver
            lines = result.stdout.split('\n')
            for line in lines:
                if 'Driver Version' in line:
                    driver_version = line.split('Driver Version: ')[1].split()[0]
                    print(f"   Versão do driver: {driver_version}")
                    break
        else:
            print("❌ Driver NVIDIA com problemas")
    except FileNotFoundError:
        print("❌ nvidia-smi não encontrado - driver NVIDIA não instalado")
    
    return True

check_cuda_installation()

# Tentar importar CuPy com tratamento de erro detalhado
try:
    import cupy as cp
    
    # Teste básico para verificar se CuPy funciona
    test_array = cp.array([1, 2, 3])
    test_result = test_array + 1
    
    CUPY_AVAILABLE = True
    print(f"✅ CuPy {cp.__version__} funcionando corretamente")
    
    # Informações detalhadas da GPU
    device_id = cp.cuda.runtime.getDevice()
    device_props = cp.cuda.runtime.getDeviceProperties(device_id)
    device_name = device_props['name'].decode()
    total_mem = cp.cuda.runtime.memGetInfo()[1]
    
    print(f"   GPU: {device_name}")
    print(f"   Memória: {total_mem // (1024**3)} GB")
    print(f"   Compute Capability: {device_props['major']}.{device_props['minor']}")
    
except ImportError as e:
    CUPY_AVAILABLE = False
    print(f"❌ CuPy não disponível: {e}")
    print("   Instale com: conda install cupy -c conda-forge")
    print("   Ou: pip install cupy-cuda11x (ou cupy-cuda12x)")

except Exception as e:
    CUPY_AVAILABLE = False
    print(f"⚠️  CuPy instalado mas com erro: {type(e).__name__}: {e}")
    
    if "nvrtc-builtins" in str(e):
        print("\n🛠️  Soluções para erro nvrtc-builtins:")
        print("1. Reinstalar CUDA Toolkit completo:")
        print("   - Baixe de https://developer.nvidia.com/cuda-toolkit")
        print("   - Instale versão compatível com seu driver")
        print("2. Ou use conda (recomendado):")
        print("   conda install cudatoolkit-dev -c conda-forge")
        print("3. Verificar PATH do CUDA:")
        print("   Adicione C:\\Program Files\\NVIDIA GPU Computing Toolkit\\CUDA\\v11.x\\bin ao PATH")
        print("4. Reiniciar VS Code após instalação")
    
    # Tentar versão CPU-only de CuPy (se existir)
    print("\n🔄 Tentando modo fallback...")

try:
    from numba import cuda
    import numba
    
    # Verificar se CUDA está disponível no Numba
    if cuda.is_available():
        NUMBA_CUDA_AVAILABLE = True
        print(f"✅ Numba CUDA {numba.__version__} disponível")
        gpu_name = cuda.get_current_device().name.decode()
        print(f"   GPU detectada: {gpu_name}")
    else:
        NUMBA_CUDA_AVAILABLE = False
        print(f"⚠️  Numba {numba.__version__} sem suporte CUDA")
        
except ImportError:
    NUMBA_CUDA_AVAILABLE = False
    print("❌ Numba não disponível")
except Exception as e:
    NUMBA_CUDA_AVAILABLE = False
    print(f"⚠️  Numba com erro: {e}")

# Configure matplotlib
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 8)

print(f"\n🖥️  Sistema: {mp.cpu_count()} núcleos CPU")
print(f"📊 NumPy: {np.__version__}")
print(f"🐍 Python: {sys.version.split()[0]}")

# Status final
if CUPY_AVAILABLE:
    print("\n🎉 GPU computing totalmente funcional!")
elif NUMBA_CUDA_AVAILABLE:
    print("\n⚠️  Apenas Numba CUDA disponível (sem CuPy)")
else:
    print("\n📚 Modo educacional: conceitos e simulações CPU")
    print("   Para funcionalidade GPU completa:")
    print("   1. Instale driver NVIDIA atualizado") 
    print("   2. Instale CUDA Toolkit")
    print("   3. Reinstale CuPy: conda install cupy -c conda-forge")

🔍 Diagnóstico CUDA:
❌ CUDA Toolkit não instalado ou não está no PATH
✅ Driver NVIDIA funcionando
   Versão do driver: 572.83
✅ CuPy 13.6.0 funcionando corretamente
   GPU: NVIDIA GeForce RTX 4050 Laptop GPU
   Memória: 5 GB
   Compute Capability: 8.9
✅ Numba CUDA 0.60.0 disponível
   GPU detectada: NVIDIA GeForce RTX 4050 Laptop GPU

🖥️  Sistema: 16 núcleos CPU
📊 NumPy: 2.0.2
🐍 Python: 3.9.23

🎉 GPU computing totalmente funcional!
✅ CuPy 13.6.0 funcionando corretamente
   GPU: NVIDIA GeForce RTX 4050 Laptop GPU
   Memória: 5 GB
   Compute Capability: 8.9
✅ Numba CUDA 0.60.0 disponível
   GPU detectada: NVIDIA GeForce RTX 4050 Laptop GPU

🖥️  Sistema: 16 núcleos CPU
📊 NumPy: 2.0.2
🐍 Python: 3.9.23

🎉 GPU computing totalmente funcional!


## 🛠️ Solução para Problemas CUDA no Windows

Se você está vendo erros como `nvrtc-builtins64_118.dll`, aqui estão as soluções:

### ✅ Método 1: Instalação via Conda (Recomendado)
```bash
# Remover instalação problemática
conda remove cupy

# Instalar versão completa com CUDA incluído
conda install cupy cudatoolkit-dev -c conda-forge

# Ou instalar versão específica para sua CUDA
conda install cupy cuda-version=11.8 -c conda-forge
```

### ✅ Método 2: CUDA Toolkit Manual
1. **Baixar CUDA Toolkit**: https://developer.nvidia.com/cuda-toolkit
2. **Instalar versão compatível** com seu driver NVIDIA
3. **Adicionar ao PATH**: `C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\vX.X\bin`
4. **Reiniciar VS Code** após instalação

### ✅ Método 3: Verificar Compatibilidade
```bash
# Verificar driver NVIDIA
nvidia-smi

# Verificar CUDA
nvcc --version

# Reinstalar CuPy para versão específica
pip uninstall cupy
pip install cupy-cuda118  # Para CUDA 11.8
```

---

## 1. Exemplo 10: Soma de Vetores com CuPy

CuPy oferece uma interface NumPy para GPUs - mudança mínima de código!

In [ ]:
# Como CuPy não está funcionando, vamos usar Numba CUDA como alternativa
if NUMBA_CUDA_AVAILABLE:
    print("🚀 Exemplo 10A: Soma de Vetores com Numba CUDA")
    print("=" * 45)
    
    @cuda.jit
    def vector_add_kernel(a, b, c):
        """Kernel CUDA para soma de vetores"""
        i = cuda.grid(1)
        if i < a.size:
            c[i] = a[i] + b[i]
    
    def vector_add_cuda(a_cpu, b_cpu):
        """Soma de vetores usando Numba CUDA"""
        N = a_cpu.size
        
        # Alocar na GPU
        start_alloc = time.perf_counter()
        a_gpu = cuda.to_device(a_cpu)
        b_gpu = cuda.to_device(b_cpu)
        c_gpu = cuda.device_array(N, dtype=np.float32)
        time_alloc = time.perf_counter() - start_alloc
        
        # Configurar grid de threads
        threads_per_block = 256
        blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
        
        # Executar kernel
        start_compute = time.perf_counter()
        vector_add_kernel[blocks_per_grid, threads_per_block](a_gpu, b_gpu, c_gpu)
        cuda.synchronize()
        time_compute = time.perf_counter() - start_compute
        
        # Transferir resultado de volta
        start_copy = time.perf_counter()
        c_cpu = c_gpu.copy_to_host()
        time_copy = time.perf_counter() - start_copy
        
        return c_cpu, time_alloc, time_compute, time_copy
    
    # Teste com diferentes tamanhos
    sizes = [1_000_000, 10_000_000]  # Reduzido para Numba
    
    for N in sizes:
        print(f"\n📊 Tamanho: {N:,} elementos")
        
        # Criar dados
        np.random.seed(42)
        a_cpu = np.random.randn(N).astype(np.float32)
        b_cpu = np.random.randn(N).astype(np.float32)
        
        # NumPy (CPU)
        start = time.perf_counter()
        c_cpu_numpy = a_cpu + b_cpu
        time_cpu = time.perf_counter() - start
        print(f"  NumPy (CPU):     {time_cpu:.4f}s")
        
        # Numba CUDA (GPU)
        c_cpu_cuda, time_alloc, time_compute, time_copy = vector_add_cuda(a_cpu, b_cpu)
        time_gpu_total = time_alloc + time_compute + time_copy
        
        # Verificar correção
        are_equal = np.allclose(c_cpu_numpy, c_cpu_cuda, rtol=1e-5)
        speedup_compute = time_cpu / time_compute
        speedup_total = time_cpu / time_gpu_total
        
        print(f"  CUDA (Compute):  {time_compute:.4f}s")
        print(f"  CUDA (Transfer): {time_alloc + time_copy:.4f}s")
        print(f"  CUDA (Total):    {time_gpu_total:.4f}s")
        print(f"  Speedup (comp):  {speedup_compute:.2f}x")
        print(f"  Speedup (total): {speedup_total:.2f}x")
        print(f"  Precisão:        {'✓' if are_equal else '✗'}")
    
    print("\n💡 Conceitos Numba CUDA:")
    print("• @cuda.jit: Compilador JIT para kernels CUDA")
    print("• cuda.grid(1): Índice global do thread")
    print("• cuda.to_device(): CPU → GPU")
    print("• .copy_to_host(): GPU → CPU")
    print("• cuda.synchronize(): Aguardar conclusão")

elif CUPY_AVAILABLE:
    # Código original do CuPy (caso funcione no futuro)
    print("🚀 Exemplo 10: Soma de Vetores com CuPy")
    print("=" * 40)
    
    sizes = [1_000_000, 10_000_000, 100_000_000]
    
    for N in sizes:
        print(f"\n📊 Tamanho: {N:,} elementos")
        
        a_cpu = np.random.randn(N).astype(np.float32)
        b_cpu = np.random.randn(N).astype(np.float32)
        
        # NumPy (CPU)
        start = time.perf_counter()
        c_cpu = a_cpu + b_cpu
        time_cpu = time.perf_counter() - start
        print(f"  NumPy (CPU):     {time_cpu:.4f}s")
        
        try:
            # CuPy (GPU)
            start_transfer = time.perf_counter()
            a_gpu = cp.asarray(a_cpu)
            b_gpu = cp.asarray(b_cpu)
            time_transfer_to = time.perf_counter() - start_transfer
            
            start = time.perf_counter()
            c_gpu = a_gpu + b_gpu
            cp.cuda.Stream.null.synchronize()
            time_gpu = time.perf_counter() - start
            
            start_transfer = time.perf_counter()
            c_gpu_cpu = cp.asnumpy(c_gpu)
            time_transfer_back = time.perf_counter() - start_transfer
            
            are_equal = np.allclose(c_cpu, c_gpu_cpu, rtol=1e-5)
            speedup_compute = time_cpu / time_gpu
            total_gpu_time = time_transfer_to + time_gpu + time_transfer_back
            speedup_total = time_cpu / total_gpu_time
            
            print(f"  CuPy (GPU):      {time_gpu:.4f}s")
            print(f"  Transferência:   {time_transfer_to + time_transfer_back:.4f}s")
            print(f"  Total GPU:       {total_gpu_time:.4f}s")
            print(f"  Speedup (comp):  {speedup_compute:.2f}x")
            print(f"  Speedup (total): {speedup_total:.2f}x")
            print(f"  Precisão:        {'✓' if are_equal else '✗'}")
            
            del a_gpu, b_gpu, c_gpu
            cp.get_default_memory_pool().free_all_blocks()
            
        except Exception as e:
            print(f"  ❌ Erro CuPy: {e}")
            break

else:
    print("? Exemplo 10: Conceitos de Soma de Vetores GPU")
    print("=" * 50)
    
    # Demonstração educacional sem GPU
    print("\n🎯 Comparação Conceitual CPU vs GPU:")
    
    sizes = [1_000_000, 10_000_000, 100_000_000]
    print("Elementos       CPU (s)    GPU (s)    Speedup    Vantagem")
    print("-" * 60)
    
    for N in sizes:
        # Simular performance baseada em características típicas
        time_cpu_sim = N * 2e-9  # ~2ns por operação CPU
        time_gpu_compute = N * 0.2e-9  # ~0.2ns por operação GPU
        time_gpu_transfer = N * 4 / (20e9)  # Transfer PCIe
        time_gpu_total = time_gpu_compute + time_gpu_transfer
        
        speedup = time_cpu_sim / time_gpu_total
        
        if speedup > 2:
            advantage = "GPU wins"
        elif speedup > 1.1:
            advantage = "Slight GPU"
        else:
            advantage = "Transfer cost"
            
        print(f"{N:>10,}    {time_cpu_sim:.4f}    {time_gpu_total:.4f}     {speedup:.2f}x     {advantage}")
    
    print("\n💡 Princípios fundamentais:")
    print("• CPU: Poucos cores (~16) otimizados para latência")
    print("• GPU: Milhares de cores (~2048) otimizados para throughput")
    print("• GPU é eficiente para operações paralelas massivas")
    print("• Transferência de dados pode ser gargalo")
    
    print("\n🔧 Anatomia de uma operação GPU:")
    print("1. **Alocar memória na GPU**")
    print("2. **Transferir dados CPU → GPU** (via PCIe)")
    print("3. **Executar kernel na GPU** (massivamente paralelo)")
    print("4. **Transferir resultados GPU → CPU**")
    print("5. **Liberar memória GPU**")
    
    print("\n📝 Exemplo de código (conceitual):")
    print("""
    # Pseudocódigo GPU
    a_gpu = allocate_gpu_memory(size)
    b_gpu = allocate_gpu_memory(size)
    c_gpu = allocate_gpu_memory(size)
    
    copy_to_gpu(a_cpu, a_gpu)    # Transfer
    copy_to_gpu(b_cpu, b_gpu)    
    
    launch_kernel(add_vectors, a_gpu, b_gpu, c_gpu)  # Parallel execution
    
    copy_to_cpu(c_gpu, c_cpu)    # Transfer back
    
    free_gpu_memory(a_gpu, b_gpu, c_gpu)
    """)

## 2. Exemplo 11: Multiplicação de Matrizes Massivas

Para matrizes grandes, GPUs mostram seu verdadeiro poder!

In [ ]:
if CUPY_AVAILABLE:
    print("🧮 Exemplo 11: Multiplicação de Matrizes Massivas")
    print("=" * 50)
    
    # Testar diferentes tamanhos de matrizes
    matrix_sizes = [1024, 2048, 4096]
    
    for size in matrix_sizes:
        print(f"n📊 Matrizes {size}x{size} (elementos: {size**2:,})")
        
        # Criar matrizes aleatórias
        print("  Gerando matrizes aleatórias...")
        np.random.seed(42)
        A_cpu = np.random.randn(size, size).astype(np.float32)
        B_cpu = np.random.randn(size, size).astype(np.float32)
        
        # NumPy (CPU) com BLAS otimizado
        print("  Executando multiplicação no CPU...")
        start = time.perf_counter()
        C_cpu = np.dot(A_cpu, B_cpu)
        time_cpu = time.perf_counter() - start
        print(f"  NumPy (CPU):     {time_cpu:.4f}s")
        
        # Transferir para GPU
        start = time.perf_counter()
        A_gpu = cp.asarray(A_cpu)
        B_gpu = cp.asarray(B_cpu)
        time_transfer_to = time.perf_counter() - start
        print(f"  Transferência→GPU: {time_transfer_to:.4f}s")
        
        # CuPy (GPU)
        print("  Executando multiplicação na GPU...")
        start = time.perf_counter()
        C_gpu = cp.dot(A_gpu, B_gpu)
        cp.cuda.Stream.null.synchronize()
        time_gpu = time.perf_counter() - start
        print(f"  CuPy (GPU):      {time_gpu:.4f}s")
        
        # Transferir resultado de volta
        start = time.perf_counter()
        C_gpu_cpu = cp.asnumpy(C_gpu)
        time_transfer_back = time.perf_counter() - start
        print(f"  Transferência←GPU: {time_transfer_back:.4f}s")
        
        # Verificar precisão
        max_error = np.max(np.abs(C_cpu - C_gpu_cpu))
        are_close = np.allclose(C_cpu, C_gpu_cpu, rtol=1e-4)
        
        # Calcular speedups
        speedup_compute = time_cpu / time_gpu
        total_gpu_time = time_transfer_to + time_gpu + time_transfer_back
        speedup_total = time_cpu / total_gpu_time
        
        # Calcular FLOPS (operações de ponto flutuante por segundo)
        flops = 2 * size**3  # Multiplicação de matrizes: 2*n³ operações
        gflops_cpu = flops / (time_cpu * 1e9)
        gflops_gpu = flops / (time_gpu * 1e9)
        
        print(f"n  📈 Performance:")
        print(f"    CPU GFLOPS:      {gflops_cpu:.2f}")
        print(f"    GPU GFLOPS:      {gflops_gpu:.2f}")
        print(f"    Speedup (comp):  {speedup_compute:.2f}x")
        print(f"    Speedup (total): {speedup_total:.2f}x")
        print(f"    Erro máximo:     {max_error:.2e}")
        print(f"    Precisão:        {'✓' if are_close else '✗'}")
        
        # Limpeza de memória GPU
        del A_gpu, B_gpu, C_gpu
        cp.get_default_memory_pool().free_all_blocks()
    
    print("n💡 Observações:")
    print("• GPU acelera significativamente operações matriciais grandes")
    print("• GFLOPS (Giga FLOPS) mede eficiência computacional")
    print("• Speedup melhora com tamanho do problema")
    print("• Essencial considerar overhead de transferência")

else:
    print("⚠️  Demonstração conceitual de multiplicação de matrizes GPU:")
    
    # Simular performance típica
    sizes = [1024, 2048, 4096]
    print("Speedups típicos para multiplicação de matrizes:")
    print("Tamanho    CPU (s)    GPU (s)    Speedup")
    print("-" * 40)
    
    for size in sizes:
        # Estimativas baseadas em hardware típico
        flops = 2 * size**3
        time_cpu_est = flops / (100e9)  # ~100 GFLOPS CPU
        time_gpu_est = flops / (2000e9)  # ~2000 GFLOPS GPU
        speedup_est = time_cpu_est / time_gpu_est
        
        print(f"{size:>6}x{size:<6} {time_cpu_est:.3f}     {time_gpu_est:.4f}    {speedup_est:.1f}x")

## 3. Introdução ao Numba CUDA

Numba CUDA permite escrever kernels customizados para máximo controle sobre a GPU.

In [ ]:
if NUMBA_CUDA_AVAILABLE:
    print("🔧 Introdução ao Numba CUDA")
    print("=" * 30)
    
    @cuda.jit
    def add_kernel(a, b, c):
        # """Kernel CUDA para soma elemento por elemento"""
        # Obter índice do thread atual
        i = cuda.grid(1)
        
        # Verificar bounds
        if i < a.size:
            c[i] = a[i] + b[i]
    
    @cuda.jit 
    def square_kernel(arr, result):
        # """Kernel CUDA para elevar ao quadrado"""
        i = cuda.grid(1)
        
        if i < arr.size:
            result[i] = arr[i] * arr[i]
    
    # Exemplo 12: Monte Carlo π com CUDA
    @cuda.jit
    def monte_carlo_pi_kernel(rng_states, n_per_thread, results):
        # """Kernel para estimativa de π por Monte Carlo"""
        thread_id = cuda.grid(1)
        
        if thread_id >= rng_states.size:
            return
            
        # Cada thread conta pontos dentro do círculo
        count = 0
        for i in range(n_per_thread):
            x = cuda.random.xoroshiro128p_uniform_float32(rng_states, thread_id)
            y = cuda.random.xoroshiro128p_uniform_float32(rng_states, thread_id)
            
            if x*x + y*y <= 1.0:
                count += 1
        
        results[thread_id] = count
    
    def monte_carlo_pi_cuda(n_samples, n_threads=256):
        # """Estimativa de π usando CUDA"""
        n_blocks = (n_samples + n_threads - 1) // n_threads
        samples_per_thread = n_samples // (n_blocks * n_threads)
        
        # Alocar arrays na GPU
        rng_states = cuda.random.create_xoroshiro128p_states(n_blocks * n_threads, seed=42)
        results = cuda.device_array(n_blocks * n_threads, dtype=np.int32)
        
        # Executar kernel
        start = time.perf_counter()
        monte_carlo_pi_kernel[n_blocks, n_threads](rng_states, samples_per_thread, results)
        cuda.synchronize()
        time_gpu = time.perf_counter() - start
        
        # Transferir resultados e somar
        results_host = results.copy_to_host()
        total_inside = np.sum(results_host)
        total_samples = samples_per_thread * n_blocks * n_threads
        
        pi_estimate = 4.0 * total_inside / total_samples
        return pi_estimate, time_gpu
    
    print("n🎲 Exemplo 12: Monte Carlo π com Numba CUDA")
    print("-" * 45)
    
    # Teste com diferentes números de amostras
    sample_counts = [1_000_000, 10_000_000, 100_000_000]
    
    for n_samples in sample_counts:
        print(f"nAmostras: {n_samples:,}")
        
        # Versão CPU para comparação
        start = time.perf_counter()
        np.random.seed(42)
        x_cpu = np.random.uniform(-1, 1, n_samples)
        y_cpu = np.random.uniform(-1, 1, n_samples)
        inside_cpu = np.sum((x_cpu**2 + y_cpu**2) <= 1)
        pi_cpu = 4 * inside_cpu / n_samples
        time_cpu = time.perf_counter() - start
        
        # Versão GPU
        pi_gpu, time_gpu = monte_carlo_pi_cuda(n_samples)
        
        speedup = time_cpu / time_gpu
        error_cpu = abs(pi_cpu - np.pi)
        error_gpu = abs(pi_gpu - np.pi)
        
        print(f"  CPU: π ≈ {pi_cpu:.6f}, erro = {error_cpu:.6f}, tempo = {time_cpu:.4f}s")
        print(f"  GPU: π ≈ {pi_gpu:.6f}, erro = {error_gpu:.6f}, tempo = {time_gpu:.4f}s")
        print(f"  Speedup: {speedup:.2f}x")
    
    print("n💡 Conceitos CUDA importantes:")
    print("• Grid: coleção de blocos")
    print("• Block: coleção de threads")
    print("• Thread: unidade de execução")
    print("• cuda.grid(1): obtém índice global do thread")
    print("• cuda.synchronize(): aguarda conclusão")

else:
    print("⚠️  Numba CUDA não disponível. Conceitos fundamentais:")
    print("""
    Hierarquia CUDA:
    
    Grid (toda a GPU)
    ├── Block 0 (grupo de threads)
    │   ├── Thread 0
    │   ├── Thread 1
    │   └── Thread N
    ├── Block 1
    └── Block M
    
    Kernel CUDA exemplo:
    
    @cuda.jit
    def meu_kernel(input_array, output_array):
        # Obter índice único para este thread
        i = cuda.grid(1)
        
        # Verificar bounds
        if i < input_array.size:
            output_array[i] = input_array[i] * 2
    
    # Lançar kernel
    threads_per_block = 256
    blocks_per_grid = (array.size + threads_per_block - 1) // threads_per_block
    meu_kernel[blocks_per_grid, threads_per_block](input_arr, output_arr)
    """)

## 4. Exemplo 13: Simulação de Difusão de Calor 2D

Vamos implementar uma simulação realística de engenharia: difusão de calor em uma placa metálica.

In [ ]:
# 🔍 Status das bibliotecas GPU
print("🧪 Verificando status GPU...")

if NUMBA_CUDA_AVAILABLE:
    print("⚠️  Numba CUDA detectado mas com problemas de execução")
    print("   Possíveis causas:")
    print("   • Conflito de versões CUDA/Driver")
    print("   • Problema de inicialização GPU")
    print("   • Limitações do Windows/Jupyter")
    print("   Continuando com modo CPU-only...")
    NUMBA_CUDA_AVAILABLE = False  # Forçar CPU-only
    
elif CUPY_AVAILABLE:
    print("⚠️  CuPy com problemas NVRTC")
    print("   Continuando com modo CPU-only...")
    CUPY_AVAILABLE = False  # Forçar CPU-only
    
else:
    print("ℹ️  Modo CPU apenas - ideal para aprendizado de conceitos")

print("🚀 Pronto para simulação educacional!")

In [ ]:
print("🌡️  Exemplo 13: Simulação de Difusão de Calor 2D")
print("=" * 50)

# Função CPU otimizada e responsiva
def heat_equation_cpu_optimized(T, alpha, dx, dy, dt, steps):
    """Versão CPU otimizada com feedback constante"""
    ny, nx = T.shape
    T_new = T.copy()
    
    print(f"🔥 Iniciando {steps} passos de simulação térmica...")
    start_time = time.perf_counter()
    
    for step in range(steps):
        # Feedback frequente para mostrar progresso
        if step % 10 == 0 or step < 5:
            elapsed = time.perf_counter() - start_time
            progress = 100 * step / steps
            print(f"   📊 Passo {step:2d}/{steps} ({progress:4.1f}%) - {elapsed:.2f}s")
        
        # Diferenças finitas vetorizadas (muito mais rápido que loops)
        T_new[1:-1, 1:-1] = T[1:-1, 1:-1] + alpha * dt * (
            # Derivada segunda em x
            (T[2:, 1:-1] - 2*T[1:-1, 1:-1] + T[:-2, 1:-1]) / (dx*dx) +
            # Derivada segunda em y  
            (T[1:-1, 2:] - 2*T[1:-1, 1:-1] + T[1:-1, :-2]) / (dy*dy)
        )
        
        # Swap arrays eficientemente
        T, T_new = T_new, T
    
    total_time = time.perf_counter() - start_time
    print(f"✅ Simulação concluída em {total_time:.3f}s")
    return T, total_time

# Simular versão GPU (conceitual) para comparação educacional
def simulate_gpu_performance(cpu_time, nx, ny, steps):
    """Simula performance típica de GPU baseada em características do problema"""
    
    # Fatores baseados em hardware real
    total_points = nx * ny * steps
    
    # GPU tem ~10x mais cores, mas overhead de setup
    if total_points < 100_000:
        # Problemas pequenos: overhead domina
        gpu_speedup = 0.8  # GPU mais lenta
        note = "GPU overhead > benefício"
    elif total_points < 1_000_000:
        # Problemas médios: speedup moderado
        gpu_speedup = 2.5
        note = "GPU começando a valer a pena"
    else:
        # Problemas grandes: GPU brilha
        gpu_speedup = 8.0
        note = "GPU clara vantagem"
    
    simulated_gpu_time = cpu_time / gpu_speedup
    
    return simulated_gpu_time, gpu_speedup, note

# Parâmetros para simulação rápida mas educativa
Lx, Ly = 1.0, 1.0  # Placa 1m x 1m
nx, ny = 50, 50    # Grid 50x50 = 2,500 pontos (bem responsivo)
dx = Lx / (nx - 1)
dy = Ly / (ny - 1)
alpha = 1e-4       # Difusividade térmica do aço
dt = 0.2 * min(dx, dy)**2 / (4 * alpha)  # Passo estável
steps = 100        # 100 passos para demonstração

print(f"📋 Configuração da Simulação:")
print(f"   🏗️  Domínio: {Lx}m × {Ly}m (placa de aço)")
print(f"   🔢 Malha: {nx}×{ny} = {nx*ny:,} pontos")
print(f"   ⏱️  Passos: {steps} (dt = {dt:.2e}s)")
print(f"   🌡️  Material: aço (α = {alpha:.0e} m²/s)")

# Condições iniciais
T_initial = np.zeros((ny, nx), dtype=np.float32)

# Fonte de calor: solda no canto
heat_size = 8
T_initial[:heat_size, :heat_size] = 200.0  # 200°C (solda)

# Condições de contorno: bordas resfriadas
T_initial[0, :] = 20.0   # Borda superior
T_initial[-1, :] = 20.0  # Borda inferior  
T_initial[:, 0] = 20.0   # Borda esquerda
T_initial[:, -1] = 20.0  # Borda direita

print(f"\n🔥 Condições Iniciais:")
print(f"   • Fonte: solda {heat_size}×{heat_size} a 200°C (canto)")
print(f"   • Bordas: resfriamento forçado a 20°C")
print(f"   • Interior: temperatura ambiente (0°C)")

# Executar simulação CPU
print(f"\n🚀 Executando Simulação CPU...")
T_final, cpu_time = heat_equation_cpu_optimized(T_initial.copy(), alpha, dx, dy, dt, steps)

# Simular performance GPU para comparação educacional
gpu_time, speedup, note = simulate_gpu_performance(cpu_time, nx, ny, steps)

print(f"\n📊 Análise de Performance:")
print(f"   💻 CPU Real: {cpu_time:.3f}s")
print(f"   🎮 GPU Simulada: {gpu_time:.3f}s") 
print(f"   ⚡ Speedup Estimado: {speedup:.1f}x")
print(f"   💡 Observação: {note}")

# Análise dos resultados
temp_max = np.max(T_final)
temp_min = np.min(T_final)
temp_center = T_final[ny//2, nx//2]

print(f"\n🌡️  Resultados Térmicos:")
print(f"   ? Temp. máxima: {temp_max:.1f}°C")
print(f"   ? Temp. mínima: {temp_min:.1f}°C") 
print(f"   🎯 Temp. centro: {temp_center:.1f}°C")
print(f"   📉 Gradiente: {temp_max - temp_min:.1f}°C")

# Visualização educacional
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Estado inicial
im1 = axes[0].imshow(T_initial, cmap='hot', origin='lower', 
                     extent=[0, Lx, 0, Ly], vmin=0, vmax=200)
axes[0].set_title('🔥 Estado Inicial\n(t = 0s)')
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
cbar1 = plt.colorbar(im1, ax=axes[0], shrink=0.8)
cbar1.set_label('Temperatura (°C)')

# Estado final
im2 = axes[1].imshow(T_final, cmap='hot', origin='lower',
                     extent=[0, Lx, 0, Ly], vmin=0, vmax=200)
axes[1].set_title(f'🌡️  Estado Final\n(t = {steps*dt:.4f}s)')
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
cbar2 = plt.colorbar(im2, ax=axes[1], shrink=0.8)
cbar2.set_label('Temperatura (°C)')

# Perfil de temperatura (linha central)
x_line = np.linspace(0, Lx, nx)
y_line = np.linspace(0, Ly, ny)
temp_horizontal = T_final[ny//2, :]  # Linha horizontal no meio
temp_vertical = T_final[:, nx//2]    # Linha vertical no meio

axes[2].plot(x_line, temp_horizontal, 'r-', linewidth=2, label='Horizontal (meio)')
axes[2].plot(y_line, temp_vertical, 'b-', linewidth=2, label='Vertical (meio)')
axes[2].set_title('📈 Perfis de Temperatura')
axes[2].set_xlabel('Posição (m)')
axes[2].set_ylabel('Temperatura (°C)')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# Comparação CPU vs GPU conceitual
print(f"\n🎯 GPU vs CPU - Análise Educacional:")
print(f"┌─────────────────┬─────────────┬─────────────┬─────────────┐")
print(f"│ Tamanho Problema│ CPU (s)     │ GPU (s)     │ Speedup     │")
print(f"├─────────────────┼─────────────┼─────────────┼─────────────┤")

# Simular diferentes tamanhos de problema
problem_sizes = [(32, 50), (64, 100), (128, 200), (256, 500)]
for (grid_size, time_steps) in problem_sizes:
    # Estimar tempo CPU baseado no resultado real
    scale_factor = (grid_size**2 * time_steps) / (nx**2 * steps)
    estimated_cpu = cpu_time * scale_factor
    
    estimated_gpu, estimated_speedup, _ = simulate_gpu_performance(
        estimated_cpu, grid_size, grid_size, time_steps
    )
    
    print(f"│ {grid_size:3d}×{grid_size:<3d} ({time_steps:3d} passos)│ {estimated_cpu:8.3f}    │ {estimated_gpu:8.3f}    │ {estimated_speedup:8.1f}x    │")

print(f"└─────────────────┴─────────────┴─────────────┴─────────────┘")

print(f"\n💡 Insights para Engenharia Civil:")
print(f"• 🏗️  **Análise térmica estrutural**: Gradientes em concreto, aço")
print(f"• 🔥 **Simulação de incêndios**: Propagação de calor em edifícios")
print(f"• ❄️  **Pontes térmicas**: Isolamento de envoltórias") 
print(f"• 🌡️  **Conforto térmico**: Distribuição de temperatura em ambientes")
print(f"• ⚡ **GPU ideal para**: Malhas grandes (>100k pontos), tempo real")

print(f"\n🚀 Próximos Passos:")
print(f"• Para usar GPU: Resolver problemas CUDA (ver células anteriores)")
print(f"• Aumentar malha: 256×256 ou maior para problemas reais")
print(f"• Métodos avançados: Elementos finitos, multigrid")
print(f"• Paralelização: MPI para clusters, múltiplas GPUs")

## 5. Integração CPU + GPU e Ferramentas Avançadas

Vamos ver como combinar CPU e GPU eficientemente e explorar ferramentas do ecossistema.

In [ ]:
print("🔗 Integração CPU + GPU e Ecossistema HPC")
print("=" * 45)

print("n📚 Ferramentas Avançadas do Ecossistema:")
print("n1. 🚀 **Dask + CuPy**: Computação distribuída em múltiplas GPUs")
print("   • Escalabilidade para clusters")
print("   • DataFrame distribuídos")
print("   • Exemplo: `import dask.array as da; da.from_array(cupy_array)`")

print("n2. 🧠 **PyTorch/TensorFlow**: Machine Learning em larga escala")
print("   • Redes neurais para previsão estrutural")
print("   • Processamento de imagens de inspeção")
print("   • Detecção de anomalias em estruturas")

print("n3. 🐳 **Singularity/Docker**: Containers para HPC")
print("   • Reprodutibilidade em clusters")
print("   • Isolamento de dependências")
print("   • Portabilidade entre sistemas")

print("n4. 📊 **Rapids**: Aceleração completa do pipeline de dados")
print("   • cuDF (pandas para GPU)")
print("   • cuML (scikit-learn para GPU)")
print("   • cuGraph (NetworkX para GPU)")

# Demonstração de pipeline híbrido CPU+GPU
def hybrid_data_processing_example():
    
    print("n💡 Exemplo de Pipeline Híbrido CPU+GPU:")
    print("n1. **Pré-processamento no CPU**:")
    print("   • Leitura de arquivos")
    print("   • Parsing de dados")
    print("   • Validação e limpeza")
    
    print("n2. **Computação intensiva na GPU**:")
    print("   • Simulações numéricas")
    print("   • Operações matriciais massivas")
    print("   • Processamento paralelo")
    
    print("n3. **Pós-processamento no CPU**:")
    print("   • Análise estatística")
    print("   • Geração de relatórios")
    print("   • Visualização")
    
    if CUPY_AVAILABLE:
        print("n🔧 Implementação prática:")
        
        # Simular dados de entrada (CPU)
        print("   Gerando dados no CPU...")
        data_cpu = np.random.randn(1000, 1000).astype(np.float32)
        
        # Transferir para GPU para processamento intensivo
        print("   Transferindo para GPU...")
        data_gpu = cp.asarray(data_cpu)
        
        # Operações intensivas na GPU
        print("   Processando na GPU...")
        start = time.perf_counter()
        
        # Simulação: múltiplas operações matriciais
        result_gpu = cp.dot(data_gpu, data_gpu.T)
        result_gpu = cp.linalg.eigvals(result_gpu)
        result_gpu = cp.sort(result_gpu)
        
        cp.cuda.Stream.null.synchronize()
        gpu_time = time.perf_counter() - start
        
        # Transferir de volta para análise no CPU
        print("   Transferindo resultado de volta...")
        result_cpu = cp.asnumpy(result_gpu)
        
        # Análise final no CPU
        print("   Analisando no CPU...")
        mean_eigenval = np.mean(result_cpu.real)
        std_eigenval = np.std(result_cpu.real)
        
        print(f"n📊 Resultados:")
        print(f"   Tempo GPU: {gpu_time:.4f}s")
        print(f"   Autovalor médio: {mean_eigenval:.6f}")
        print(f"   Desvio padrão: {std_eigenval:.6f}")
    
    else:
        print("n⚠️  CuPy não disponível para demonstração prática")

hybrid_data_processing_example()

print("n🎯 Boas Práticas para CPU+GPU:")
print("n• **Minimize transferências**: Mantenha dados na GPU o máximo possível")
print("• **Sobreposição**: Use streams CUDA para computação e transferência simultâneas")
print("• **Pinned memory**: Acelera transferências CPU↔GPU")
print("• **Batch processing**: Processe múltiplos itens juntos")
print("• **Profiling**: Use nvprof, Nsight para identificar gargalos")

print("n📈 Quando usar GPU vs CPU:")
print("n**Use GPU quando:**")
print("• Alto paralelismo (milhares de operações simultâneas)")
print("• Operações matemáticas intensivas")
print("• Dados regulares e homogêneos")
print("• Pouca ramificação condicional")

print("n**Use CPU quando:**")
print("• Código sequencial complexo")
print("• Muitas ramificações condicionais")
print("• I/O intensivo")
print("• Processamento de strings/texto")

print("n🔮 Futuro da Computação Heterogênea:")
print("• **SYCL/DPC++**: Programação unificada CPU/GPU/FPGA")
print("• **Intel oneAPI**: Toolchain multi-arquitetura")
print("• **AMD HIP**: Portabilidade CUDA→ROCm")
print("• **WebGPU**: Computação GPU no navegador")

# Exemplo de medição de throughput
if CUPY_AVAILABLE:
    print("n📊 Benchmark Final: Throughput de Dados")
    print("-" * 40)
    
    data_sizes = [1, 10, 100]  # MB
    
    for size_mb in data_sizes:
        n_elements = (size_mb * 1024 * 1024) // 4  # 4 bytes por float32
        
        # CPU
        data_cpu = np.random.randn(n_elements).astype(np.float32)
        start = time.perf_counter()
        result_cpu = np.sum(data_cpu**2)
        time_cpu = time.perf_counter() - start
        throughput_cpu = size_mb / time_cpu
        
        # GPU
        data_gpu = cp.asarray(data_cpu)
        start = time.perf_counter()
        result_gpu = cp.sum(data_gpu**2)
        cp.cuda.Stream.null.synchronize()
        time_gpu = time.perf_counter() - start
        throughput_gpu = size_mb / time_gpu
        
        print(f"{size_mb:3d} MB: CPU {throughput_cpu:6.1f} MB/s, GPU {throughput_gpu:6.1f} MB/s")